# Hong Kong residential data: a few lines to explore

## Goal
Use the built-in groups in **hk-property-data 0.6.0+** to see an overview and read numeric DataFrames by group and date. No copied group dictionary or helper function is needed. The package reads the existing GitHub publication; it does not run collectors or create new economic estimates.

[Grouped indicator guide](https://github.com/Hogan-cyber/Real-Estate-Data#指标组速查) · [Published dataset](https://github.com/Hogan-cyber/Real-Estate-Data/releases/tag/residential-data-latest)

## Setup
Use Python 3.10+. Install once below; restart the kernel after upgrading an existing session. Skip installation on subsequent visits. Saved outputs are the snapshot printed by the next cell; run again for your current session.

In [ ]:
import sys
assert sys.version_info >= (3, 10), "Select a Python 3.10+ kernel."
%pip install --quiet --disable-pip-version-check --upgrade "https://github.com/Hogan-cyber/Real-Estate-Data/releases/download/residential-data-latest/hk_property_data-0.6.3-py3-none-any.whl"

## Steps
### 1. Overview and full DataFrame
`overview()` gives one row per group and frequency; `catalog()` explains individual IDs and includes their group. `history()` returns numeric columns on a date index. Missing cells are retained. `...` only abbreviates the display; `all_data` holds the complete result.

`D/M/Q/A/E` mean daily/monthly/quarterly/annual/effective events. Overview records include original missing observations; available values exclude them. Dates show the union of member record ranges, not guaranteed non-missing coverage.

In [ ]:
import hk_property_data as hk

try:
    from IPython.display import display
except ImportError:
    display = print

market = hk.Residential()
overview = market.overview()
catalog = market.catalog()
all_data = market.history(labels="id")

print("Package:", hk.__version__)
print("Snapshot generated:", all_data.attrs["data_generated_at"])
print(f"{len(catalog)} indicators; {len(all_data):,} date rows")
display(overview)
display(all_data)

### 2. Group and date selection
Change the group name and date arguments. `end=None` includes all subsequently available dates; `end="2025-12"` includes that whole month. Default columns use readable names; use `labels="id"` for stable IDs.

In [ ]:
group_name = "Price indices"
df = market.history(group=group_name, start="2025-01", end=None)
display(market.catalog(group=group_name)[["metric_name", "unit", "frequency", "first_date", "last_date"]])
display(df)

### 3. Multiple groups and frequency
Group lists form a union. Frequency filters select existing observations; they never calculate monthly averages from daily rates. All these calls reuse the same session cache.

In [ ]:
prices_and_rents = market.history(group=["Price indices", "Rent indices"], start="2025-01")
display(prices_and_rents.tail(3))
monthly_hibor = market.history(group="HIBOR", frequency="M", start="2025-01")
display(monthly_hibor.tail(3))

### 4. Suffixes and existing indicator IDs
Related class and district variants remain independently selectable. Only actual catalogue IDs are sent to the reader.

In [ ]:
prices = market.catalog(group="Average prices")
kowloon_ids = prices.index[prices.index.str.endswith("_kowloon")].tolist()
kowloon = market.history(series=kowloon_ids, start="2025-01")
display(kowloon.tail(3))
selected = market.history(series=["rvd_1_4_all_classes", "rvd_1_3_all_classes"],
                          start="2025-01", end="2025-12", labels="id")
display(selected.tail(3))

### 5. Original source records
Per-record sources, flags and notes remain in `observations()`. The built-in Mortgage estimates group contains existing formula-based results and their input shares, not official money-lender observations.

In [ ]:
source = market.observations(group="Bank mortgages", series="hk_mortgage_drawn_amount",
                             start="2025-01", end="2025-01")
display(source[["date", "metric_name", "value", "unit", "status", "source_url"]])

## Checks
These checks confirm numeric tables and complete group coverage. They do not independently reconcile every value with official source files.

In [ ]:
import pandas as pd
assert isinstance(all_data.index, pd.DatetimeIndex)
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in all_data.dtypes)
assert overview["indicators"].sum() == len(catalog) == all_data.shape[1]
assert overview["records"].sum() == catalog["observations"].sum()
assert set(market.catalog(group="HIBOR", frequency="M")["frequency"]) == {"M"}
print("Numeric DataFrame, group coverage and frequency selection checked.")

## Next Steps
Inspect `overview`, `catalog`, `all_data` or `df` directly in Jupyter / Spyder. You can bind a group once: `prices = hk.Residential(group="Price indices")`, then call `prices.history(start="2025-01")`.

Refresh explicitly, then rerun other reads to replace their older DataFrames:
```python
overview = market.overview(refresh=True)
all_data = market.history(labels="id")
```
The shared Arrow cache is capped at 8 MiB; returned DataFrames use additional memory. [Python guide](https://github.com/Hogan-cyber/Real-Estate-Data/blob/main/PYTHON_PACKAGE.md)
